# Visualization Colab Notebook

## 0. Runtime Setup
### 0.1 Configure Visualization Inputs

What this step does: defines the repository revision, fixed Colab roots, manifest/sample selection scope, sample selection mode, render mode, and output controls for this visualization run.

Required inputs: operator-edited REPO_URL, REPO_REF, SELECTION_SPLITS, RENDER_MODE, SELECTION_MODE, SAMPLE_ID, RANDOM_SEED, OUTPUT_EXISTS_POLICY, and OUTPUT_FILENAME values. WORKTREE_ROOT and DRIVE_PROJECT_ROOT are fixed Colab project roots and should only change if the project-wide Colab layout changes.

Produced outputs: configured constants printed for review before Drive, repository, or artifact restore operations run. SELECTION_SPLITS defines the manifest/sample selection scope: sample_id lookup and random selection are restricted to these splits, the selected sample split determines which processed sample archive is restored, and side-by-side mode restores the raw BFH archive only for the selected sample split. SELECTION_SPLITS accepts any non-empty official split subset when required artifacts exist: ("train",), ("val",), ("test",), ("train", "val"), ("train", "test"), ("val", "test"), or ("train", "val", "test").

When this step may be skipped: only when these constants are already defined in the current runtime exactly as intended for this visualization.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/0xmillennium/text-to-sign-production.git"
REPO_REF = "master"
SELECTION_SPLITS = ("val",)
RENDER_MODE = "skeleton_only"  # "skeleton_only" or "side_by_side"
SELECTION_MODE = "random"  # "sample_id" or "random"
SAMPLE_ID = ""
RANDOM_SEED = 13
OUTPUT_EXISTS_POLICY = "overwrite"  # "fail", "overwrite", or "skip"
OUTPUT_FILENAME = "output.mp4"

WORKTREE_ROOT = Path("/content/text-to-sign-production")
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/text-to-sign-production")
print("Worktree root:", WORKTREE_ROOT)
print("Drive project root:", DRIVE_PROJECT_ROOT)
print("Selection split scope:", SELECTION_SPLITS)
print("Render mode:", RENDER_MODE)
print("Selection mode:", SELECTION_MODE)
print("Output filename:", OUTPUT_FILENAME)


### 0.2 Mount Google Drive

What this step does: mounts Google Drive at /content/drive and confirms the mirrored project root can be addressed under MyDrive.

Required inputs: DRIVE_PROJECT_ROOT from step 0.1 and an authenticated Colab Drive session.

Produced outputs: a mounted Drive filesystem with DRIVE_PROJECT_ROOT.parent present.

When this step may be skipped: only when Drive is already mounted and DRIVE_PROJECT_ROOT.parent has been verified in this runtime.


In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
if not DRIVE_PROJECT_ROOT.parent.is_dir():
    raise FileNotFoundError(f"Drive MyDrive root is missing: {DRIVE_PROJECT_ROOT.parent}")
print("Drive mounted:", DRIVE_PROJECT_ROOT.parent)


### 0.3 Ensure zstd Is Available

What this step does: checks for the zstd command and installs the Debian package when the Colab runtime does not provide it.

Required inputs: Colab apt package access and sudo permission in the runtime.

Produced outputs: a runtime where zstd --version succeeds before any archive command is used.

When this step may be skipped: only when shutil.which("zstd") already finds zstd and zstd --version has succeeded in this runtime.


In [ ]:
import shutil

if shutil.which("zstd") is None:
    !sudo apt-get update

    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("Failed to update apt package index.")

    !sudo apt-get install -y zstd

    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("Failed to install zstd.")
    print("Installed zstd.")
else:
    print("zstd is already available.")

!zstd --version

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("zstd command is not available after preflight.")


### 0.4 Acquire Repository

What this step does: removes any old runtime checkout, clones REPO_URL into WORKTREE_ROOT, checks out REPO_REF, and prints the resolved commit.

Required inputs: REPO_URL, REPO_REF, writable /content, and network access to the repository.

Produced outputs: a fresh repository checkout at WORKTREE_ROOT with the requested revision checked out.

When this step may be skipped: only when WORKTREE_ROOT already contains the same requested revision and the operator intentionally wants to reuse it.


In [ ]:
%cd /content/
!rm -rf {WORKTREE_ROOT}

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError(f"Failed to remove existing directory: {WORKTREE_ROOT}")

!git clone {REPO_URL} {WORKTREE_ROOT}

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("Failed to clone repository.")

!git -C {WORKTREE_ROOT} checkout {REPO_REF}

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError(f"Failed to checkout revision {REPO_REF}.")

print(f"Repository cloned at {WORKTREE_ROOT}")

!git -C {WORKTREE_ROOT} rev-parse HEAD

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("Failed to determine checked out revision.")

print("Repository ready:", WORKTREE_ROOT)


### 0.5 Install Python Dependencies

What this step does: runs pip from the repository checkout and installs the Colab requirements file.

Required inputs: WORKTREE_ROOT containing requirements-colab.txt and Python package index access.

Produced outputs: the notebook runtime has the project Colab Python dependencies installed.

When this step may be skipped: only when these exact requirements have already been installed in the active Colab runtime.


In [ ]:
%cd {WORKTREE_ROOT}

%pip install --upgrade pip

%pip install -r "requirements-colab.txt"

print("Repository requirements installed. Please, restart the session.")


### 0.6 Add Source Tree To Python Path

What this step does: changes to the repository checkout and prepends the local src tree to sys.path for notebook imports.

Required inputs: WORKTREE_ROOT with a src directory from the checked-out repository.

Produced outputs: WORKTREE_ROOT / "src" is available on sys.path for subsequent imports.

When this step may be skipped: only when the same src path is already present on sys.path in this runtime.


In [ ]:
import sys

%cd {WORKTREE_ROOT}

if str(WORKTREE_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(WORKTREE_ROOT / "src"))

print("Repository src directory added to sys.path:", WORKTREE_ROOT / "src")


### 0.7 Import Workflow API

What this step does: imports only the Visualization workflow API that owns runtime layout, artifact restore specs, render planning, publish plans, and verifier functions.

Required inputs: the installed dependencies and sys.path entry from steps 0.5 and 0.6.

Produced outputs: workflow functions used by later cells; no project core, data, artifacts, visualization domain, or modeling modules are imported by the notebook.

When this step may be skipped: only when these exact workflow imports already exist in the current runtime.


In [ ]:
from text_to_sign_production.workflows.visualization import (
    build_visualization_publish_plan,
    build_visualization_render_plan,
    build_visualization_runtime_plan,
    build_visualization_sample_plan,
    run_visualization_workflow,
    summarize_visualization_publish_plan,
    summarize_visualization_sample,
    summarize_visualization_workflow_outputs,
    validate_visualization_inputs,
    verify_visualization_runtime_inputs,
)

print("Visualization workflow API ready:", build_visualization_runtime_plan.__module__)


## 1. Restore Required Manifests
### 1.1 Build Visualization Runtime Plan

What this step does: builds the workflow-owned runtime plan and direct manifest file restore specs for the selected manifest/sample scope.

Required inputs: WORKTREE_ROOT, DRIVE_PROJECT_ROOT, SELECTION_SPLITS, RENDER_MODE, OUTPUT_EXISTS_POLICY, and Drive-published direct manifest files for every requested selection split.

Produced outputs: visualization_runtime_plan with verified Drive manifest paths, runtime target paths, byte counts, and visible command specs.

When this step may be skipped: only when visualization_runtime_plan already reflects the current roots, selection split scope, render settings, and Drive manifest files.


In [ ]:
visualization_runtime_plan = build_visualization_runtime_plan(
    project_root=WORKTREE_ROOT,
    drive_project_root=DRIVE_PROJECT_ROOT,
    splits=SELECTION_SPLITS,
    render_mode=RENDER_MODE,
    output_exists_policy=OUTPUT_EXISTS_POLICY,
)
print("Runtime project root:", visualization_runtime_plan.project_root)
print("Drive project root:", visualization_runtime_plan.drive_project_root)
print("Visualization run name:", visualization_runtime_plan.run_name)
print("Visualization run root:", visualization_runtime_plan.run_root)
print("Visualization outputs root:", visualization_runtime_plan.outputs_root)
print("Selection split scope:", visualization_runtime_plan.splits)
print("Manifest files to restore:", len(visualization_runtime_plan.manifest_files))
for label, spec in visualization_runtime_plan.manifest_files.items():
    print(f"- {label}: {spec.source_path} -> {spec.target_path} ({spec.expected_input_bytes} bytes)")


### 1.2 Copy Required Manifest Files

What this step does: executes each workflow-provided visible direct manifest restore command with byte progress.

Required inputs: visualization_runtime_plan.manifest_files and writable runtime data directories.

Produced outputs: restored processed and filtered manifest files under the runtime checkout.

When this step may be skipped: only when the same manifest files have already been copied into this checkout and will be verified in the next step.


In [ ]:
%cd {WORKTREE_ROOT}
for label, restore_spec in visualization_runtime_plan.manifest_files.items():
    spec = restore_spec.restore_command
    print(spec.bash)
    !bash -o pipefail -c {spec.arg}

    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(spec.failure)

    print(f"{label} restored:", restore_spec.target_path)


### 1.3 Verify Manifest Restore

What this step does: asks the workflow verifier to check that every workflow-declared manifest file is present, non-empty, and byte-size matched after direct restore.

Required inputs: visualization_runtime_plan and restored manifest files.

Produced outputs: restored_manifest_paths mapping each workflow-declared manifest file to a verified runtime path.

When this step may be skipped: only when restored_manifest_paths has already been verified after the current direct restore.


In [ ]:
runtime_input_summary = verify_visualization_runtime_inputs(visualization_runtime_plan)
restored_manifest_paths = runtime_input_summary.manifest_paths
for label, manifest_path in restored_manifest_paths.items():
    print(f"{label}:", manifest_path)


## 2. Select And Restore Processed Sample
### 2.1 Configure Sample Selection

What this step does: normalizes operator selection inputs into the primitive values passed to the workflow sample planner.

Required inputs: SELECTION_MODE, SAMPLE_ID, RANDOM_SEED, and SELECTION_SPLITS from step 0.1 plus restored workflow-declared manifest files.

Produced outputs: configured_sample_id and configured_random_seed values for workflow-owned sample planning. sample_id lookup and random selection are restricted to SELECTION_SPLITS.

When this step may be skipped: only when these configured values already represent the intended exact sample or deterministic random selection.


In [ ]:
configured_sample_id = SAMPLE_ID.strip() or None
configured_random_seed = RANDOM_SEED
print("Selection split scope:", SELECTION_SPLITS)
print("Selection mode:", SELECTION_MODE)
print("Sample id:", configured_sample_id)
print("Random seed:", configured_random_seed)


### 2.2 Build Visualization Sample Plan

What this step does: selects one sample from restored workflow-declared manifests and builds selected processed sample and optional raw video restore specs.

Required inputs: visualization_runtime_plan, configured_sample_id, configured_random_seed, and processed manifests from section 1; processed manifests must exist before sample selection runs.

Produced outputs: sample_plan and sample_restore_spec with selected split, sample path, archive restore spec, and optional raw BFH restore spec for side-by-side rendering. If a sample_id is not found inside SELECTION_SPLITS, the workflow raises an error for the selected scope.

When this step may be skipped: only when sample_plan already reflects the same manifest set and selection inputs.


In [ ]:
sample_plan = build_visualization_sample_plan(
    visualization_runtime_plan,
    selection_mode=SELECTION_MODE,
    sample_id=configured_sample_id,
    random_seed=configured_random_seed,
)
sample_restore_spec = sample_plan.sample_archive
print("Selected sample:", sample_plan.sample_id)
print("Selected split:", sample_plan.split)
print("Processed sample archive:", sample_restore_spec.archive_path)


### 2.3 Inspect Selected Sample

What this step does: prints the workflow-owned selected sample context before restoring or rendering the selected sample.

Required inputs: sample_plan plus the primitive selection and render controls from step 0.1.

Produced outputs: sample_summary with sample id, split, text when available, selection mode, random seed, processed sample path, source video path, render mode, archive paths/sizes, and raw restore requirement.

When this step may be skipped: only when the selected sample context has already been inspected for this exact sample_plan.


In [ ]:
sample_summary = summarize_visualization_sample(
    sample_plan,
    selection_mode=SELECTION_MODE,
    random_seed=configured_random_seed,
    render_mode=RENDER_MODE,
)

print("Selected Sample")
print("---------------")
print("sample_id:", sample_summary.sample_id)
print("split:", sample_summary.split)
print("text:", sample_summary.text if sample_summary.text else "unavailable")
print("selection mode:", sample_summary.selection_mode)
print("random seed:", sample_summary.random_seed if sample_summary.random_seed is not None else "n/a")
print("processed sample path:", sample_summary.sample_path)
print("source video path:", sample_summary.source_video_path if sample_summary.source_video_path else "unavailable")
print("render mode:", sample_summary.render_mode)
print(
    "processed sample archive:",
    f"{sample_summary.processed_archive_path} ({sample_summary.processed_archive_size} bytes)",
)
if sample_summary.raw_restore_required:
    print("raw BFH restore: required")
    print(
        "raw BFH archive:",
        f"{sample_summary.raw_archive_path} ({sample_summary.raw_archive_size} bytes)",
    )
else:
    print("raw BFH restore: skipped")


### 2.4 Extract Selected Processed Sample Archive

What this step does: executes the workflow-provided visible selected split sample archive extraction command with byte progress.

Required inputs: sample_restore_spec, zstd support, and writable WORKTREE_ROOT.

Produced outputs: restored processed sample files for the selected split under the runtime checkout.

When this step may be skipped: only when the selected split sample archive has already been extracted into this checkout and will be verified in the next step.


In [ ]:
%cd {WORKTREE_ROOT}
spec = sample_restore_spec.extract
print(spec.bash)
!bash -o pipefail -c {spec.arg}

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError(spec.failure)

print("Selected processed sample archive extracted:", sample_restore_spec.archive_path)


### 2.5 Verify Selected Processed Sample

What this step does: asks the workflow verifier to check that the selected processed .npz sample exists and is non-empty after archive extraction.

Required inputs: sample_plan and restored selected split processed sample files.

Produced outputs: selected_sample_path pointing at the verified runtime processed sample file.

When this step may be skipped: only when selected_sample_path has already been verified after the current sample archive extraction.


In [ ]:
runtime_input_summary = verify_visualization_runtime_inputs(
    visualization_runtime_plan,
    sample_plan,
    verify_raw_video=False,
)
selected_sample_path = runtime_input_summary.selected_sample_path
print("Selected processed sample restored:", selected_sample_path)


## 3. Restore Optional Raw Video
### 3.1 Extract Raw BFH Archive When Required

What this step does: executes the workflow-provided visible raw BFH source archive extraction command when side-by-side rendering needs source video.

Required inputs: sample_plan.raw_video, zstd support, and writable runtime raw BFH directories.

Produced outputs: restored raw BFH source archive contents for side-by-side mode, or a clear skip for skeleton-only mode.

When this step may be skipped: when RENDER_MODE is skeleton_only, because skeleton-only rendering does not require raw BFH video restore. Side-by-side mode requires restoring the selected raw BFH source archive. Raw BFH archives remain source-format archives and are verified after extraction.


In [ ]:
if sample_plan.raw_video is None:
    print("Skeleton-only render selected; raw BFH extraction skipped.")
else:
    raw_video_restore_spec = sample_plan.raw_video
    %cd {WORKTREE_ROOT}
    spec = raw_video_restore_spec.extract
    print(spec.bash)
    !bash -o pipefail -c {spec.arg}

    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(spec.failure)

    print("Selected raw BFH archive extracted:", raw_video_restore_spec.extraction_root)


### 3.2 Verify Source Video When Required

What this step does: asks the workflow verifier to check side-by-side raw BFH restore outputs and selected source video, or skip cleanly for skeleton-only mode.

Required inputs: sample_plan and optional raw BFH archive contents restored in step 3.1.

Produced outputs: selected_source_video_path for side-by-side mode, or None for skeleton-only mode.

When this step may be skipped: when RENDER_MODE is skeleton_only, because no source video is rendered.


In [ ]:
runtime_input_summary = verify_visualization_runtime_inputs(
    visualization_runtime_plan,
    sample_plan,
)
selected_source_video_path = runtime_input_summary.selected_source_video_path
if selected_source_video_path is None:
    print("Skeleton-only render selected; source video verification skipped.")
else:
    print("Source video restored:", selected_source_video_path)


## 4. Render Visualization
### 4.1 Build Visualization Render Plan

What this step does: builds the workflow-owned render plan from the selected sample and primitive notebook controls. The canonical visualization output is an OpenCV-generated MP4 using mp4v. It is intended as the project visualization artifact and should play after download or from Drive in standard media players. Colab inline playback may not work in all browser/runtime combinations.

Required inputs: sample_plan, RENDER_MODE, OUTPUT_EXISTS_POLICY, and OUTPUT_FILENAME.

Produced outputs: render_plan with selected sample, render mode, output path, and primitive render settings.

When this step may be skipped: only when render_plan already reflects the same selected sample and render controls.


In [ ]:
render_plan = build_visualization_render_plan(
    sample_plan,
    render_mode=RENDER_MODE,
    output_exists_policy=OUTPUT_EXISTS_POLICY,
    output_filename=OUTPUT_FILENAME,
)
print("Visualization render output:", render_plan.output_path)
print("Visualization render mode:", render_plan.mode)


### 4.2 Validate Visualization Inputs

What this step does: runs the workflow input validator before rendering.

Required inputs: render_plan, restored selected processed sample, and optional source video for side-by-side mode. The selected sample archive must be restored before render validation.

Produced outputs: a completed validation check for the configured visualization workflow inputs.

When this step may be skipped: only when the same render_plan has already passed validation after the current input restore.


In [ ]:
validate_visualization_inputs(render_plan)
print("Visualization inputs validated:", render_plan.sample_id)


### 4.3 Run Visualization Workflow

What this step does: renders the selected sample to one canonical runtime MP4 using the workflow-owned render plan.

Required inputs: validated render_plan and restored runtime inputs. The renderer uses the internal canonical mp4v codec and fails clearly if OpenCV cannot create the MP4.

Produced outputs: visualization_result containing the rendered runtime MP4 and render metadata.

When this step may be skipped: only when visualization_result already reflects the same selected sample, render settings, and code revision.


In [ ]:
visualization_result = run_visualization_workflow(render_plan)
print("Visualization render complete:", visualization_result.output_path)


### 4.4 Verify Visualization Output

What this step does: asks the workflow verifier to check and summarize the rendered visualization output.

Required inputs: visualization_result from the completed render step.

Produced outputs: visualization_summary with sample id, split, mode, codec, output path, and output byte size.

When this step may be skipped: only when these result postconditions have already been checked for this exact render.


In [ ]:
visualization_summary = summarize_visualization_workflow_outputs(visualization_result)

print("Visualization Output")
print("--------------------")
print("sample_id:", visualization_summary.sample_id)
print("split:", visualization_summary.split)
print("mode:", visualization_summary.mode)
print("codec:", visualization_summary.codec)
print("runtime output:", visualization_summary.output_path)
print("size:", visualization_summary.output_size)
print("render plan metadata:", visualization_summary.render_plan_metadata_path)
print("render result metadata:", visualization_summary.render_result_metadata_path)


### 4.5 Display Visualization Output

What this step does: displays the canonical rendered MP4 inline in the notebook for visual inspection.

Required inputs: visualization_summary.output_path from the verified render step.

Produced outputs: an inline IPython video display attempt for the rendered visualization output. Colab inline playback may not work in all browser/runtime combinations; if inline playback fails, use the printed runtime path or published Drive copy. A playback failure here does not necessarily mean rendering failed.

When this step may be skipped: only when the same rendered output has already been displayed and inspected in this notebook run.


In [ ]:
from IPython.display import Video, display

print("Runtime output:", visualization_summary.output_path)
print("Codec:", visualization_summary.codec)
display(Video(str(visualization_summary.output_path), embed=True))


## 5. Publish Visualization Output
### 5.1 Build Visualization Publish Plan

What this step does: builds the workflow-owned publish plan for copying the verified runtime MP4 to the mirrored Drive visualization artifact location.

Required inputs: visualization_summary and DRIVE_PROJECT_ROOT.

Produced outputs: publish_plan with source path, target path, byte count, and visible byte-progress copy command.

When this step may be skipped: only when publish_plan already targets the intended Drive output for this verified render.


In [ ]:
publish_plan = build_visualization_publish_plan(
    visualization_result,
    drive_project_root=DRIVE_PROJECT_ROOT,
)
publish_specs = (publish_plan.output_spec, *publish_plan.metadata_specs)
print("Publish run root:", publish_plan.drive_run_root)
for publish_spec in publish_specs:
    print("Publish source:", publish_spec.source_path)
    print("Publish target:", publish_spec.target_path)
    print("Publish bytes:", publish_spec.expected_input_bytes)


### 5.2 Copy Visualization Output To Drive

What this step does: executes the workflow-provided visible publish command with byte progress.

Required inputs: publish_plan and writable Drive visualization artifact directory.

Produced outputs: a copied visualization MP4 at publish_plan.target_path.

When this step may be skipped: only when the Drive MP4 already matches the current rendered runtime output and will be verified in the next step.


In [ ]:
for publish_spec in publish_specs:
    spec = publish_spec.publish_command
    print(spec.bash)
    !bash -o pipefail -c {spec.arg}

    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(spec.failure)

    print("Visualization artifact copied to Drive:", publish_spec.target_path)


### 5.3 Verify Published Visualization Output

What this step does: asks the workflow verifier to check that the Drive-published visualization MP4 exists and matches the runtime output byte size, then prints a compact publish summary.

Required inputs: publish_plan and the copied Drive MP4.

Produced outputs: published_visualization_path and visualization_publish_summary pointing at the verified Drive visualization MP4.

When this step may be skipped: only when the same Drive MP4 has already been verified after the current copy step.


In [ ]:
visualization_publish_summary = summarize_visualization_publish_plan(publish_plan)

print("Visualization Output")
print("--------------------")
print("sample_id:", visualization_summary.sample_id)
print("split:", visualization_summary.split)
print("mode:", visualization_summary.mode)
print("codec:", visualization_summary.codec)
print("runtime output:", visualization_publish_summary.output_source_path)
print("size:", visualization_publish_summary.output_size)
print("published output:", visualization_publish_summary.output_target_path)
for label, metadata_path in visualization_publish_summary.metadata_paths.items():
    print(f"published {label}:", metadata_path)
